# Trimorious 2.0 | Scientific Playground
Inference only. No training, publishing or changes to existing weights. Start a clean T4 GPU runtime, run all cells, authorize Drive and enter a private playground password when prompted. The Gradio share URL is Internet-accessible; authentication is required. Do not enter confidential research data. Stop the last cell or disconnect to close the app.

The exact base revision and adapter fingerprints come from the final audit. The default searches your project folder for matching weights; use Hugging Face mode if local files are unavailable. A single loaded model serves both Trimorious and the unmodified Qwen3 baseline, with the adapter disabled for the baseline. Comparison mode uses the same single-turn prompt and deterministic settings, not a formal benchmark.

In [ ]:
# 1. Dependencies (fresh runtime recommended)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.6', 'peft==0.18.1', 'accelerate==1.15.0',
    'bitsandbytes==0.50.2', 'huggingface_hub==0.36.2', 'safetensors==0.8.0',
    'gradio>=6,<7'])
print('Dependencies installed. If Colab requests a restart, restart and run all again.')

In [ ]:
# 2. Configuration and adapter integrity
from pathlib import Path
import hashlib, json, os
from google.colab import drive
drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/trimorious_PRIVATE_m1_4_1')
SOURCE = 'drive' # @param ["drive", "huggingface"]
ADAPTER_PATH = '' # Optional absolute path to final adapter directory
BASE = 'Qwen/Qwen3-4B-Instruct-2507'
REV = 'cdbee75f17c01a7cc42f958dc650907174af0554'
REPO = 'roime81/trimorious-research-4B-v2'
EXPECTED = {'adapter_model.safetensors': '5ce2eadc1ca1f33d28b2a6a517eaad53a73263441c5519c3dc662590514a609d', 'adapter_config.json': '181bb2231889bb9049b78892c907f96103493933055dfb3a4027d393c772dff9'}

def sha(path):
    with path.open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()

def matches(p):
    return all((p/name).is_file() and sha(p/name)==digest for name,digest in EXPECTED.items())

if SOURCE == 'drive':
    if ADAPTER_PATH:
        adapter = Path(ADAPTER_PATH)
    else:
        assert PROJECT.is_dir(), 'Project folder not mounted.'
        candidates = sorted({p.parent for p in PROJECT.rglob('adapter_config.json')}, key=lambda p: ('release_package' not in str(p), str(p)))
        adapter = next((p for p in candidates if matches(p)), None)
        assert adapter is not None, 'No matching final adapter in Drive. Set ADAPTER_PATH or SOURCE=huggingface.'
elif SOURCE == 'huggingface':
    from google.colab import userdata
    from huggingface_hub import snapshot_download, HfApi
    token = userdata.get('HF_TOKEN')
    assert token, 'Add HF_TOKEN as a Colab secret with read access.'
    commit = HfApi().model_info(REPO, token=token).sha
    adapter = Path(snapshot_download(REPO, revision=commit, token=token, allow_patterns=list(EXPECTED)))
    del token
    print('Hub adapter commit:', commit)
else:
    raise ValueError('Unknown SOURCE')
assert matches(adapter), 'Adapter hashes differ from the audited release. Stop; do not substitute weights.'
config = json.loads((adapter/'adapter_config.json').read_text())
assert config['base_model_name_or_path']==BASE and config['revision']==REV
print('Audited adapter files: verified')


In [ ]:
# 3. Load one 4-bit base + adapter (no training)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
assert torch.cuda.is_available(), 'Select T4 GPU in Runtime settings.'
free,total = torch.cuda.mem_get_info()
print(f'GPU: {torch.cuda.get_device_name(0)}; free {free/2**30:.2f} GiB')
assert free >= 8*2**30, 'Insufficient free memory. Stop other workloads or restart runtime.'
q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(BASE,revision=REV,trust_remote_code=False)
base = AutoModelForCausalLM.from_pretrained(BASE,revision=REV,quantization_config=q,
    device_map={'':0},torch_dtype=torch.float16,low_cpu_mem_usage=True,trust_remote_code=False)
assert base.is_loaded_in_4bit
model=PeftModel.from_pretrained(base,str(adapter),is_trainable=False).eval()
print('Trimorious adapter loaded; no second model needed.')

In [ ]:
# 4. Inference functions and smoke test
import time, threading
from contextlib import nullcontext
lock=threading.Lock()

def generate(messages, baseline=False, max_new_tokens=256, temperature=0.0):
    prompt=tok.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    inputs=tok(prompt,return_tensors='pt',add_special_tokens=False).to(model.get_input_embeddings().weight.device)
    count=inputs['input_ids'].shape[1]
    if count>2048:
        raise ValueError('Context exceeds 2048 input tokens. Clear chat or shorten your prompt.')
    kwargs=dict(max_new_tokens=int(max_new_tokens),do_sample=temperature>0,pad_token_id=tok.eos_token_id,use_cache=True)
    if temperature>0: kwargs.update(temperature=float(temperature),top_p=0.9)
    with lock, torch.inference_mode(), (model.disable_adapter() if baseline else nullcontext()):
        torch.cuda.synchronize(); start=time.perf_counter()
        out=model.generate(**inputs,**kwargs)
        torch.cuda.synchronize(); elapsed=time.perf_counter()-start
    answer=tok.decode(out[0,count:],skip_special_tokens=True)
    return answer, f'{out.shape[1]-count} tokens · {elapsed:.2f}s'

answer,stats=generate([{'role':'user','content':'What is 7 multiplied by 8?'}],max_new_tokens=64)
print('Inference smoke test:',answer,stats)
assert answer.strip(), 'Empty generation.'


In [ ]:
# 5. Launch the password-protected playground
import gradio as gr
import getpass, secrets
os.environ['GRADIO_ANALYTICS_ENABLED']='False'

def chat(message, history, mode, max_tokens, temperature):
    try:
        if mode=='Compare both (single turn)':
            msgs=[{'role':'user','content':message}]
            v,vs=generate(msgs,False,max_tokens,0.0)
            b,bs=generate(msgs,True,max_tokens,0.0)
            return f'### Trimorious 2.0\n{v}\n\n_{vs}_\n\n---\n### Qwen3 foundation\n{b}\n\n_{bs}_'
        msgs=[]
        for item in history:
            if item.get('role') not in ('user','assistant'): continue
            content=item.get('content')
            if not isinstance(content,str): raise ValueError('Text-only chat. Clear the conversation.')
            # Gradio displays metrics after answers; remove that suffix before reuse.
            if item['role']=='assistant' and '\n\n---\n_' in content:
                content=content.rsplit('\n\n---\n_',1)[0]
            msgs.append({'role':item['role'],'content':content})
        msgs.append({'role':'user','content':message})
        answer,stats=generate(msgs,mode=='Qwen3 foundation',max_tokens,temperature)
        return answer+'\n\n---\n_'+stats+'_'
    except ValueError as e:
        return str(e)

password=getpass.getpass('Set a private playground password (at least 12 characters): ')
assert len(password)>=12, 'Use a stronger password.'
# Run outside the mounted Drive and block its file-serving path.
app_dir=Path('/content/trimorious_gradio_app'); app_dir.mkdir(exist_ok=True); os.chdir(app_dir)
with gr.Blocks(title='Trimorious 2.0 | Scientific Playground') as demo:
    gr.Markdown('# Trimorious 2.0 — Scientific Playground\nExperimental research adapter. Verify answers independently. Clear chat before changing models. Comparison mode ignores history and uses greedy decoding.')
    with gr.Row():
        mode=gr.Radio(['Trimorious 2.0','Qwen3 foundation','Compare both (single turn)'],value='Trimorious 2.0',label='Model')
        max_tokens=gr.Slider(64,512,value=256,step=64,label='Maximum response tokens')
        temp=gr.Slider(0,1,value=0,step=0.1,label='Temperature (comparison uses 0)')
    gr.ChatInterface(fn=chat,additional_inputs=[mode,max_tokens,temp],concurrency_limit=1)
print('Login username: researcher. Use the password you just entered. Keep the share URL private.')
demo.queue(default_concurrency_limit=1).launch(share=True,auth=('researcher',password),blocked_paths=['/content/drive',str(adapter)],show_error=False)
